In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load artifacts
best_xgb  = joblib.load('xgb_best_model.pkl')
scaler    = joblib.load('scaler.pkl')
feat_names = joblib.load('feature_names.pkl')

# Load processed data
df = pd.read_csv('Edited_Data.csv')
X  = df.drop('Churn', axis=1)
y  = df['Churn']

X_scaled_df = pd.DataFrame(scaler.transform(X), columns=feat_names)
churn_probs = best_xgb.predict_proba(X_scaled_df)[:, 1]
df['churn_probability'] = churn_probs

Steps for AB testing:

1. A/B Test
   
2. p-value
   
3. Is the difference statistically significant?
   
4. Effect size
   
5. How large is the difference?
   
6.  ROI

In [2]:
#AB test for customer retention intervention
# These are the customers who qualify for the retention experiment.

target_customers = df[
    (df['churn_probability'] >= 0.60) &
    (df['Contract_Month-to-month'] == 1)
].copy()

print("Target customers:", len(target_customers))

Target customers: 911


In [3]:
# Creating experimental and control groups for the AB test

#Control → no retention offer
#Treatment → receives retention offer

import numpy as np

np.random.seed(42)

target_customers['group'] = np.random.choice(
    ['Control', 'Treatment'],
    size=len(target_customers),
    p=[0.5, 0.5]
)

print("Control group:", len(target_customers[target_customers['group'] == 'Control']))
print("Treatment group:", len(target_customers[target_customers['group'] == 'Treatment']))

Control group: 456
Treatment group: 455


In [4]:
# df['churn']:s historical churn. It happened before we assigned customers to our hypothetical treatment/control groups.
# Therefore, it cannot be used as evidence of the effect of the treatment.

# Create the post-intervention outcome
# We need to simulate this for demo purposes.

np.random.seed(42)

control_mask = target_customers['group'] == 'Control'
treatment_mask = target_customers['group'] == 'Treatment'

target_customers.loc[control_mask, 'churned_after_intervention'] = np.random.binomial(
    1,
    0.40,
    control_mask.sum()
)

target_customers.loc[treatment_mask, 'churned_after_intervention'] = np.random.binomial(
    1,
    0.32,
    treatment_mask.sum()
)

In [5]:
# calculate churn rates for control and treatment groups
# Control: no special retention intervention. We observe their normal churn behavior.
# Treatment: receives the retention intervention. We observe whether the intervention changes their churn behavior.

churn_rates = target_customers.groupby('group')[
    'churned_after_intervention'
].mean()

print(churn_rates)

group
Control      0.414474
Treatment    0.290110
Name: churned_after_intervention, dtype: float64


In [6]:
#Statisticaal test

# We will use a two-sided two-proportion z-test to determine if there is a statistically significant difference in churn rates between the control and treatment groups.
# Since the outcome is binary (0 = did not churn, 1 = churned) and we are comparing two independent groups, a two-sided two-proportion z-test is appropriate.

# two groups: control and treatment

control = target_customers[target_customers['group'] == 'Control']

treatment = target_customers[target_customers['group'] == 'Treatment']

In [7]:
n_control = len(control)
n_treatment = len(treatment)

churn_control = control['churned_after_intervention'].sum()
churn_treatment = treatment['churned_after_intervention'].sum()

print("Control:", n_control, "customers,", churn_control, "churned")
print("Treatment:", n_treatment, "customers,", churn_treatment, "churned")

Control: 456 customers, 189.0 churned
Treatment: 455 customers, 132.0 churned


In [8]:
# churn rates for control and treatment groups
# churn_control/ncontrol

churn_rate_control = churn_control / n_control
churn_rate_treatment = churn_treatment / n_treatment

print("Churn rate (Control):", churn_rate_control)
print("Churn rate (Treatment):", churn_rate_treatment)

Churn rate (Control): 0.4144736842105263
Churn rate (Treatment): 0.29010989010989013


In [9]:
#setting up the hypothesis
# Null hypothesis(H0): the churn rate for control and treatment groups is the same.
#p_control = p_treatment


# Alternative hypothesis(H1): the churn rate for control and treatment groups is different.
#p_control != p_treatment



In [10]:
# Ztest for two-proportion

from statsmodels.stats.proportion import proportions_ztest

count = [churn_control, churn_treatment] # no. of churned customers
nobs = [n_control, n_treatment] #no. of observations
z_stat, p_value = proportions_ztest(
    count=count,
    nobs=nobs,
    alternative='two-sided'
)


print("Z-statistic:", z_stat)
print("P-value:", p_value)

Z-statistic: 3.9288255838753776
P-value: 8.536172159173836e-05


In [11]:
# Interpreting the p value
#if p_value < 0.05:
    #print("Reject the null hypothesis. There is a significant difference in churn rates between the control and treatment groups.")
#else:
    #print("Fail to reject the null hypothesis. There is no significant difference in churn rates between the control and treatment groups.")

alpha = 0.05
if p_value < alpha:
    print("Reject H0: The churn rates are significantly different.")
else:
    print("Fail to reject H0: There is not enough evidence that the churn rates are different.")

Reject H0: The churn rates are significantly different.


In [12]:
# Quantify the difference in churn rates between the control and treatment groups

difference_in_churn_rates = churn_rate_control - churn_rate_treatment
print("Difference in churn rates between control and treatment groups:", difference_in_churn_rates)


Difference in churn rates between control and treatment groups: 0.1243637941006362


In [13]:
# This means the difference in churn rates between the control and treatment groups is 12 percentage points.        
# The Treatment group had an 12 percentage-point lower churn rate in our simulated experiment.

In [14]:
#relative reduction in churn rate

relative_reduction_in_churn_rate = difference_in_churn_rates / churn_rate_control
print("Relative reduction in churn rate:", relative_reduction_in_churn_rate)

Relative reduction in churn rate: 0.30005232862375714


In [15]:
#absolute reduction in churn rate: 12 percentage points
#and the relative reduction in churn rate: 30%

In [16]:
#Estimated customers retained
#for ex: if Treatment has 500 customers, (500*12) would give the estimated number of customers retained (fewer churns)
effect = churn_rate_control - churn_rate_treatment
customer_retained = int(n_treatment) * effect
print (str(round(customer_retained)) + " customers are estimated to be retained")


#Calculate revenue retained
avg_monthy_churn_revenue = treatment['MonthlyCharges'].mean()

#monthly revenue retained
monthly_revenue_retained = customer_retained * avg_monthy_churn_revenue

#annual revenue retained
annual_revenue_retained = monthly_revenue_retained * 12

print(
    f"Estimated monthly revenue retained: "
    f"${monthly_revenue_retained:,.0f}"
)
print(
    f"Estimated annual revenue retained: "
    f"${annual_revenue_retained:,.0f}"
)

57 customers are estimated to be retained
Estimated monthly revenue retained: $4,483
Estimated annual revenue retained: $53,800


In [17]:
# Calculating Intervention cost
# Assume the intervention cost $50/customer
intervention_cost = 50

#retention offer costs to all treatment customers:
total_intervention_cost = n_treatment * intervention_cost
print(
    f"Total intervention cost: "
    f"${total_intervention_cost:,.0f}"
)


Total intervention cost: $22,750


ROI = (Benefit - Cost) / Cost

In [18]:
roi = (annual_revenue_retained - total_intervention_cost)/ total_intervention_cost 
profit = annual_revenue_retained - total_intervention_cost
print(
    f"Estimated ROI: "
    f"{roi:,.2f}"
)
print(
    f"Estimated profit: "
    f"${profit:,.0f}"
)

Estimated ROI: 1.36
Estimated profit: $31,050


Conclusion: Extimated ROI: 136% based on AB test outcome and assumed intervention cost.


In [19]:
# estimated ROI 136%
# if intervention cost = 10,000
# benefit total = 23000
# profit = $13,600 i.e(23000 - 10000)

In [ ]:
# print("A/B Test Summary")
# print("-" * 30)

# print(f"Control churn rate:   {churn_rate_control:.2%}")
# print(f"Treatment churn rate: {churn_rate_treatment:.2%}")
# print(f"Churn reduction:      {effect:.2%}")
# print(f"P-value:              {p_value:.4f}")
# print(f"Estimated ROI:        {roi:.2%}")

A/B Test Summary
------------------------------
Control churn rate:   41.45%
Treatment churn rate: 29.01%
Churn reduction:      12.44%
P-value:              0.0001
Estimated ROI:        136.48%
